# 02a - Inference and Permutation

## The analytical question

Lecture 01b found that median annual grocery spending is 9,706 source monetary units higher for the recorded Retail clients than for the recorded Horeca clients. This notebook makes three parts of the resulting evidence visible: the statistic chosen to represent the question, the structure preserved and removed by permutation, and the comparison between the observed statistic and outcomes under the no-association model.

### Learning goals

By the end of this notebook, you should be able to:

- compare statistics that answer different questions about the same groups;
- identify what a permutation preserves and which relationship it removes;
- interpret an observed statistic relative to a permutation distribution;
- assess whether a proposed reassignment preserves required data structure.

## Load the recorded case

The UCI *Wholesale customers* data set contains annual spending records for 440 clients of a wholesale distributor. `Channel` records an existing Horeca or Retail classification. `Grocery` records annual spending in source monetary units.

Data source: Cardoso, M. (2013). *Wholesale customers* [Dataset]. UCI Machine Learning Repository. <https://doi.org/10.24432/C5030X>. The source archive is available under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import permutation_test

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/292/wholesale%2Bcustomers.zip"

customers = pd.read_csv(DATA_URL, compression="zip")
customers["channel"] = customers["Channel"].map({1: "Horeca", 2: "Retail"})

retail_grocery = customers.loc[customers["channel"] == "Retail", "Grocery"]
horeca_grocery = customers.loc[customers["channel"] == "Horeca", "Grocery"]

customers[["channel", "Grocery"]].head()

## Compare statistics for one question

The broad question “How does grocery spending differ by channel?” does not determine one calculation. The next cell computes four statistics from the same records. Each preserves a different aspect of the group difference.

In [ ]:
SPENDING_THRESHOLD = 10_000

statistics = pd.DataFrame(
    [
        {
            "Statistic": "Difference in means",
            "Question": "How does average spending differ?",
            "Observed value": retail_grocery.mean() - horeca_grocery.mean(),
            "Units": "source monetary units",
        },
        {
            "Statistic": "Difference in medians",
            "Question": "How does typical spending differ?",
            "Observed value": retail_grocery.median() - horeca_grocery.median(),
            "Units": "source monetary units",
        },
        {
            "Statistic": "Difference in exceedance rates",
            "Question": f"How often does spending exceed {SPENDING_THRESHOLD:,}?",
            "Observed value": 100
            * (
                retail_grocery.gt(SPENDING_THRESHOLD).mean()
                - horeca_grocery.gt(SPENDING_THRESHOLD).mean()
            ),
            "Units": "percentage points",
        },
        {
            "Statistic": "Difference in upper quartiles",
            "Question": "How does high-end spending differ?",
            "Observed value": retail_grocery.quantile(0.75)
            - horeca_grocery.quantile(0.75),
            "Units": "source monetary units",
        },
    ]
)

statistics.round({"Observed value": 1})

The calculations do not compete to reveal one hidden answer. The mean and median describe different centers. The exceedance rate represents an operational threshold. The upper quartile emphasizes the high-spending portion of each group.

### Pause: choose a statistic

Suppose warehouse capacity changes only when annual grocery spending exceeds 10,000 source monetary units. Which statistic directly represents that decision? Name one aspect of spending that the statistic discards.

##### Answer

The difference in exceedance rates directly represents how often clients in each channel cross the capacity threshold. It discards the magnitude of every value: 9,999 and 1 are both below the threshold, while 10,001 and 100,000 are both above it.

## Inspect one reassignment

The no-association model states that grocery values are unrelated to the channel labels. Under this model, the recorded label–value alignment carries no information about grocery spending.

One reassignment keeps all 440 grocery values and the 142/298 group sizes. It removes the recorded alignment between each value and its channel label.

In [ ]:
comparison = customers[["channel", "Grocery"]].copy()

demo_rng = np.random.default_rng(7130)
comparison["reassigned_channel"] = demo_rng.permutation(
    comparison["channel"].to_numpy()
)

comparison.head(12)

In [ ]:
pd.DataFrame(
    {
        "recorded labels": comparison["channel"].value_counts(),
        "reassigned labels": comparison["reassigned_channel"].value_counts(),
    }
).sort_index()

The grocery values and group sizes are unchanged. The label attached to a particular value can change. This is the computational consequence of the no-association model for this comparison.

In [ ]:
def median_difference(x, y):
    return np.median(x) - np.median(y)


observed_difference = median_difference(retail_grocery, horeca_grocery)

reassigned_retail = comparison.loc[
    comparison["reassigned_channel"] == "Retail", "Grocery"
]
reassigned_horeca = comparison.loc[
    comparison["reassigned_channel"] == "Horeca", "Grocery"
]
reassigned_difference = median_difference(reassigned_retail, reassigned_horeca)

print(f"Observed median difference: {observed_difference:,.0f}")
print(f"One reassigned median difference: {reassigned_difference:,.0f}")

One reassignment supplies one outcome under the model. A permutation distribution requires many valid reassignments.

## Build the permutation distribution

SciPy repeats the reassignment 5,000 times and calculates the declared median difference after each one. The resulting permutation distribution shows differences compatible with the no-association model and this reassignment procedure.

In [ ]:
permutation_result = permutation_test(
    (retail_grocery, horeca_grocery),
    statistic=median_difference,
    permutation_type="independent",
    alternative="two-sided",
    n_resamples=5_000,
    rng=np.random.default_rng(7130),
)

null_differences = permutation_result.null_distribution

print(f"Observed median difference: {permutation_result.statistic:,.0f}")
print(f"Simulated p-value: {permutation_result.pvalue:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.hist(null_differences, bins=40, color="#d9d9d9", edgecolor="white")
ax.axvline(
    permutation_result.statistic,
    color="#9e480e",
    linewidth=2.5,
    label="Observed difference",
)
ax.axvline(-permutation_result.statistic, color="#9e480e", linewidth=2.5)
ax.set_title("Median differences under the no-association model")
ax.set_xlabel("Retail minus Horeca median difference")
ax.set_ylabel("Number of reassignments")
ax.legend()
plt.show()

The observed difference lies beyond the simulated null differences in either direction. The p-value of 0.0004 means the observed result is very unusual under this no-association model at the available simulation resolution. Extra decimal places would not add evidence.

### Pause: assess exchangeability

Consider three proposed reassignments:

1. Move grocery values across the two fixed channel labels while preserving the group sizes.
2. For paired before-and-after measurements, reassign individual measurements freely across all conditions.
3. For weekly sales, shuffle individual weeks freely across the full time period.

Which proposal fits its data structure? For each invalid proposal, identify the structure that must remain intact.

##### Answer

The first proposal fits the stated no-association model if the recorded clients are otherwise comparable for this purpose. Paired measurements must remain paired; a valid reassignment can swap conditions within each pair. Weekly sales ordinarily require a process that preserves relevant temporal dependence rather than freely shuffling individual weeks.

## State the evidence conditionally

The observed Retail–Horeca median difference is 9,706 source monetary units. It is difficult to reconcile with the no-association model for these recorded clients at the simulation resolution used.

That result supports taking the recorded association seriously under the declared statistic and procedure. It does not establish that channel causes spending, document a sampling design for broad population inference, or determine whether the difference changes a business decision.

### Pause: complete the claim

Write two sentences:

1. State the observed result and the evidence against the no-association model.
2. Name one important question the permutation result leaves unresolved.

##### Answer

In these 440 recorded clients, the Retail median exceeds the Horeca median by 9,706 source monetary units, a result that is very unusual under the stated no-association model at the available simulation resolution. The comparison does not show that channel caused the difference or that the result generalizes to another population or period.

## Next evidence question

Permutation asks how unusual the observed statistic would be if the target relationship were absent under the stated model. Lecture 02b asks a different question: how might the observed statistic vary under a stated sampling-resampling assumption?

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the
[INSY 7130 course-materials README](../../README.md).